```text
Now that a TCP connection is established and the two machines agree to talk, they need a shared language. If the client sends raw bytes representing Hello, what does the server do with that?

This is where HTTP (Hypertext Transfer Protocol) comes in. At its core (specifically HTTP/1.1), HTTP is nothing more than a standardized format of plain text sent over that exact TCP socket we just built.

## 1. The Anatomy of an HTTP Request
When your browser or a tool like curl makes a request to a server, it takes the data, formats it into a specific text structure, and streams it over the TCP connection.

The structure has three distinct parts, separated by Carriage Return and Line Feed (\r\n):

* The Request Line: The method (GET, POST), the path (/users), and the protocol version.

* The Headers: Key-value pairs providing metadata (Host, User-Agent, Content-Type).

* The Blank Line: A crucial \r\n\r\n that tells the server, "The headers are done; the body is next."

* The Body (Optional): The actual payload (like a JSON payload in a POST request).

```text
GET / HTTP/1.1\r\n
Host: 127.0.0.1:8080\r\n
User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)...\r\n
Accept: text/html\r\n
\r\n

## 2. The Anatomy of an HTTP Response
When the server receives that text, it parses it, figures out what to do, and sends back a response formatted in the exact same paradigm.

* The Status Line: The protocol version, a numeric status code (200, 404, 500), and a status message.

* The Headers: Metadata about the response (Content-Type, Content-Length).

* The Blank Line: Again, the mandatory \r\n\r\n.

* The Body: The actual HTML, JSON, or image data being returned.

```text
HTTP/1.1 200 OK\r\n
Content-Type: text/html; charset=UTF-8\r\n
Content-Length: 46\r\n
Connection: close\r\n
\r\n
<html><body><h1>Hello, World!</h1></body></html>

## 3. Building a Web Server from Scratch (Using Raw Sockets)
Let's prove that HTTP is just text over a TCP socket. We will write a raw Python TCP server. It will wait for a connection, print the raw HTTP request text it receives from your browser, and send back a raw HTTP response string.

In [ ]:
import socket

# 1. Create the TCP Socket
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
# (Allows us to restart the script quickly without waiting for the port to free up)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# 2. Bind and Listen
server.bind(('127.0.0.1', 8080))
server.listen(1)
print("Listening on http://127.0.0.1:8080 ...")

while True:
    # 3. Accept the TCP connection (The 3-way handshake happens here)
    client_connection, client_address = server.accept()
    
    # 4. Read the incoming bytes and decode them into text
    request_data = client_connection.recv(1024).decode('utf-8')
    
    print("\n--- RAW HTTP REQUEST RECEIVED ---")
    print(request_data)
    print("---------------------------------")
    
    # 5. Construct the raw HTTP Response string
    # Notice the \r\n used for line breaks and the \r\n\r\n before the body!
    http_response = (
        "HTTP/1.1 200 OK\r\n"
        "Content-Type: text/html; charset=UTF-8\r\n"
        "Connection: close\r\n"
        "\r\n"
        "<html><body><h1 style='color: blue;'>Hello from a Raw TCP Socket!</h1></body></html>\r\n"
    )
    
    # 6. Send the text back over the TCP stream as bytes
    client_connection.sendall(http_response.encode('utf-8'))
    
    # 7. Close the connection
    client_connection.close()

Listening on http://127.0.0.1:8080 ...

--- RAW HTTP REQUEST RECEIVED ---
GET / HTTP/1.1
Host: 127.0.0.1:8080
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:154.0) Gecko/20100101 Firefox/154.0
Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8
Accept-Language: en-US,en;q=0.9
Accept-Encoding: gzip, deflate, br, zstd
Connection: keep-alive
Upgrade-Insecure-Requests: 1
Sec-Fetch-Dest: document
Sec-Fetch-Mode: navigate
Sec-Fetch-Site: none
Sec-Fetch-User: ?1
Priority: u=0, i


---------------------------------

--- RAW HTTP REQUEST RECEIVED ---
GET /favicon.ico HTTP/1.1
Host: 127.0.0.1:8080
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:154.0) Gecko/20100101 Firefox/154.0
Accept: image/avif,image/webp,image/png,image/svg+xml,image/*;q=0.8,*/*;q=0.5
Accept-Language: en-US,en;q=0.9
Accept-Encoding: gzip, deflate, br, zstd
Connection: keep-alive
Referer: http://127.0.0.1:8080/
Sec-Fetch-Dest: image
Sec-Fetch-Mode: no-cors
Sec-Fetch-Site: same-origin
Pr